###### Imports and Settings

In [1]:
import pandas as pd
#import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
pd.options.mode.chained_assignment = None  # default='warn'

###### Functions

In [2]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains Decennial SF3 2000 variables.

In [3]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [4]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

In [52]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

In [151]:
qlaces = ['1600000US4702180', #Ashland City: Cheatham
          '1600000US4739660', #Kingston Springs: Cheatham
          '1600000US4757480', #Pegram: Cheatham
          '1600000US4759560', #Pleasant View: Cheatham
          '1600000US4704620', #Belle Meade: Davidson
          '1600000US4705140', #Berry Hill: Davidson
          '1600000US4727020', #Forest Hills: Davidson
          '1600000US4729920', #Goodlettsville: Davidson/Sumner
          '1600000US4752006', #Nashville-Davidson metropolitan government (balance): Davidson
          '1600000US4754780', #Oak Hill: Davidson
          '1600000US4763140', #Ridgetop: Davidson/Robertson
          '1600000US4709880', #Burns: Dickson
          '1600000US4713080', #Charlotte: Dickson
          '1600000US4720620', #Dickson: Dickson
          '1600000US4769080', #Slayden: Dickson
          '1600000US4776860', #Vanleer: Dickson
          '1600000US4779980', #White Bluff: Dickson
          '1600000US4724320', #Erin: Houston
          '1600000US4773460', #Tennessee Ridge: Houston/Stewart
          '1600000US4744840', #McEwen: Humphreys
          '1600000US4752820', #New Johnsonville: Humphreys
          '1600000US4778560', #Waverly: Humphreys
          '1600000US4716540', #Columbia: Maury
          '1600000US4751080', #Mount Pleasant: Maury
          '1600000US4770580', #Spring Hill: Maury/Williamson
          '1600000US4715160', #Clarksville: Montgomery
          '1600000US4700200', #Adams: Robertson
          '1600000US4711980', #Cedar Hill: Robertson
          '1600000US4716980', #Coopertown: Robertson
          '1600000US4718420', #Cross Plains: Robertson
          '1600000US4730960', #Greenbrier: Robertson
          '1600000US4748980', #Millersville: Robertson/Sumner
          '1600000US4760280', #Portland: Robertson/Sumner
          '1600000US4770500', #Springfield: Robertson
          '1600000US4780200', #White House: Robertson/Sumner
          '1600000US4722360', #Eagleville: Rutherford
          '1600000US4741200', #La Vergne: Rutherford
          '1600000US4751560', #Murfreesboro: Rutherford
          '1600000US4769420', #Smyrna: Rutherford
          '1600000US4718820', #Cumberland City: Stewart
          '1600000US4721400', #Dover: Stewart
          '1600000US4728540', #Gallatin: Sumner
          '1600000US4733280', #Hendersonville: Sumner
          '1600000US4779420', #Westmoreland: Sumner
          '1600000US4708280', #Brentwood: Williamson
          '1600000US4725440', #Fairview: Williamson
          '1600000US4727740', #Franklin: Williamson
          '1600000US4753460', #Nolensville: Williamson
          '1600000US4773900', #Thompson's Station: Williamson
          '1600000US4741520', #Lebanon: Wilson
          '1600000US4750780', #Mount Juliet: Wilson
          '1600000US4778320'] #Watertown: Wilson

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 586 as of 7/5/2022.**

### SF3

In [275]:
dataguide = pd.read_csv('../data guides/DATA GUIDE SF3.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [276]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]

In [278]:
# ONE 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
onepull = one
print('Okay Finished')

Okay Finished


In [279]:
one = onepull

In [282]:
# TWO 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
twopull = two
print('Okay Finished')

Okay Finished


In [283]:
two = twopull

In [284]:
# THREE 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg3['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg3['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
three = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
three = three.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
three = three.append(state, ignore_index = True)
threepull = three
print('Okay Finished')

Okay Finished


In [285]:
three = threepull

In [286]:
# FOUR 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg4['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg4['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
four = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
four = four.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
four = four.append(state, ignore_index = True)
fourpull = four
print('Okay Finished')

Okay Finished


In [287]:
four = fourpull

In [288]:
# FIVE 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg5['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg5['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
five = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
five = five.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
five = five.append(state, ignore_index = True)
fivepull = five
print('Okay Finished')

Okay Finished


In [289]:
five = fivepull

In [290]:
# SIX 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg6['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg6['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
six = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
six = six.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
six = six.append(state, ignore_index = True)
sixpull = six
print('Okay Finished')

Okay Finished


In [291]:
six = sixpull

In [292]:
# SEVEN 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg7['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg7['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seven = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
seven = seven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seven = seven.append(state, ignore_index = True)
sevenpull = seven
print('Okay Finished')

Okay Finished


In [293]:
seven = sevenpull

In [294]:
# EIGHT 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg8['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg8['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eight = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
eight = eight.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eight = eight.append(state, ignore_index = True)
eightpull = eight
print('Okay Finished')

Okay Finished


In [295]:
eight = eightpull

In [296]:
# NINE 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg9['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg9['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
nine = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
nine = nine.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
nine = nine.append(state, ignore_index = True)
ninepull = nine
print('Okay Finished')

Okay Finished


In [297]:
nine = ninepull

In [298]:
# TEN 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg10['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg10['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
ten = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
ten = ten.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
ten = ten.append(state, ignore_index = True)
tenpull = ten
print('Okay Finished')

Okay Finished


In [299]:
ten = tenpull

In [300]:
# ELEVEN 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg11['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg11['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eleven = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
eleven = eleven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eleven = eleven.append(state, ignore_index = True)
elevenpull = eleven
print('Okay Finished')

Okay Finished


In [301]:
eleven = elevenpull

In [302]:
# TWELVE 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg12['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg12['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twelve = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
twelve = twelve.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twelve = twelve.append(state, ignore_index = True)
twelvepull = twelve
print('Okay Finished')

Okay Finished


In [303]:
twelve = twelvepull

In [304]:
# THIRTEEN 2000 SF3
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg13['SF3 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg13['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    #data = data.loc[data['CountyFIPS'].isin(GNRC)]
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
thirteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
thirteen = thirteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf3?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
thirteen = thirteen.append(state, ignore_index = True)
thirteenpull = thirteen
print('Okay Finished')

Okay Finished


In [305]:
thirteen = thirteenpull

In [306]:
last = thirteen

In [307]:
dfs = [two,three,four,five,six,seven,eight,nine,ten,eleven,twelve]

In [308]:
for df in dfs:
    df = df.drop(columns = ['NAME','StateFIPS','GeoFIPS'], inplace = True)

In [309]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

In [310]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [311]:
dfs = [one, df_merged, last]
data = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [312]:
data.head()

,NAME,GEO_ID,poverty_total_bysexbyage_series,poverty_belowlevel,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhincome_median,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%ownercost50+_wmortgage,housingcost_total%ownercostwomortgage_series,housingcost_%ownercost30to34.9_womortgage,housingcost_%ownercost35to39.9_womortgage,housingcost_%ownercost40to49.9_womortgage,housingcost_%ownercost50+_womortgage,housingcost_total_rent%hhincome_series,housingcost_%rentercost30to34.9,housingcost_%rentercost35to39.9,housingcost_%rentercost40to49.9,housingcost_%rentercost50+,housingcost_medvalue_ownerocc,housingcost_mediangrossrent_renteroccupied,structures_total_yearbuilt_series,structures_built1999to2000,structures_built1995to1998,structures_built1990to1994,structures_built1980to1989,structures_built1970to1979,structures_built1960to1969,structures_built1950to1959,structures_built1940to1949,structures_built1939orearlier,structures_medianyearbuilt,commute_total_meansoftransportationtowork_series,commute_cartruckvan,commute_cartruckvan_drovealone,commute_cartruckvan_carpooled,commute_publictransportation,commute_publictransportation_bus,commute_publictransportation_subwayorelevatedrail,commute_publictransportation_longdistancetrainorcommuterrail,commute_publictransportation_lightrailstreetcarortrolley,commute_publictransportation_ferryboat,commute_bicycle,commute_walk,commute_taxicab,commute_motorcycle,commute_othermeans,commute_workedfromhome,vehicles_tenurebyvehicles_total_series,vehicles_ownerocc_novehicle,vehicles_ownerocc_1vehicle,vehicles_ownerocc_2vehicles,vehicles_ownerocc_3vehicles,vehicles_ownerocc_4vehicles,vehicles_ownerocc_5+vehicles,vehicles_renterocc_novehicle,vehicles_renterocc_1vehicle,vehicles_renterocc_2vehicles,vehicles_renterocc_3vehicles,vehicles_renterocc_4vehicles,vehicles_renterocc_5+vehicles,foreignborn_total,fb_europe,fb_eur_northern,fb_eur_n_ireland,fb_eur_n_uk,fb_eur_n_other,fb_eur_western,fb_eur_w_austria,fb_eur_w_france,fb_eur_w_germany,fb_eur_w_netherlands,fb_eur_w_other,fb_eur_southern,fb_eur_s_greece,fb_eur_s_italy,fb_eur_s_portugal,fb_eur_s_spain,fb_eur_s_other,fb_eur_eastern,fb_eur_e_belarus,fb_eur_e_bosniaandherzegovina,fb_eur_e_czechoslovakia,fb_eur_e_hungary,fb_eur_e_poland,fb_eur_e_romania,fb_eur_e_russia,fb_eur_e_ukraine,fb_eur_e_other,fb_eur_nec,fb_asia,fb_as_eastern,fb_as_e_china,fb_as_e_chinanohongkongortaiwan,fb_as_e_hongkong,fb_as_e_taiwan,fb_as_e_japan,fb_as_e_korea,fb_as_e_other,fb_as_southcentral,fb_as_sc_afghanistan,fb_as_sc_bangladesh,fb_as_sc_india,fb_as_sc_iran,fb_as_sc_pakistan,fb_as_sc_other,fb_as_southeastern,fb_as_se_cambodia,fb_as_se_indonesia,fb_as_se_laos,fb_as_se_malaysia,fb_as_se_philippines,fb_as_se_thailand,fb_as_se_vietnam,fb_as_se_other,fb_as_western,fb_as_w_armenia,fb_as_w_iraq,fb_as_w_israel,fb_as_w_jordan,fb_as_w_lebanon,fb_as_w_syria,fb_as_w_turkey,fb_as_w_other,fb_as_nec,fb_africa,fb_af_eastern,fb_af_e_ethiopia,fb_af_e_other,fb_af_middle,fb_af_northern,fb_af_n_egypt,fb_af_n_other,fb_af_southern,fb_af_s_southafrica,fb_af_s_other,fb_af_western,fb_af_w_ghana,fb_af_w_nigeria,fb_af_w_sierraleone,fb_af_w_other,fb_af_nec,fb_oceania,fb_oc_australianewzealandsubregion,fb_oc_australianewzealandsubregion_australia,fb_oc_australianewzealandsubregion_other,fb_oc_micronesia,fb_oc_nec,fb_americas,fb_am_latinamerica,fb_am_la_caribbean,fb_am_la_cbb_barbados,fb_am_la_cbb_cuba,fb_am_la_cbb_dominicanrepublic,fb_am_la_cbb_haiti,fb_am_la_cbb_jamaica,fb_am_la_cbb_trinidadandtobago,fb_am_la_cbb_other,fb_am_la_centralamerica,fb_am_la_centr_costarica,fb_am_la_centr_elsalvador,fb_am_la_centr_guatemala,fb_am_la_centr_honduras,fb_am_la_centr_mexico,fb_am_la_centr_nicar

In [313]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500000US47021,0500000US47147,050000

In [314]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [315]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [316]:
data = transp

In [317]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [318]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [319]:
data.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee,GNRC,MPO
poverty_total_bysexbyage_series,12262.0,130178.0,7882.0,17713.0,42503.0,35399.0,53740.0,128815.0,546390.0,87757.0,7093.0,125759.0,175907.0,68074.0,562.0,3388.0,2935.0,658.0,23724.0,1361.0,308.0,1015.0,99464.0,31980.0,2998.0,1355.0,319.0,12120.0,1345.0,446.0,1342.0,5951.0,4787.0,41055.0,22389.0,13650.0,5177.0,40246.0,2758.0,18883.0,19619.0,1713.0,4959.0,12169.0,4270.0,64334.0,522169.0,1874.0,3211.0,4437.0,2145.0,2889.0,8604.0,1127.0,155.0,24906.0,13679.0,7827.0,1364.0,1220.0,357.0,1331.0,4061.0,2044.0,2284.0,7212.0,5539896.0,1371398.0,1186442.0
poverty_belowlevel,1526.0,12982.0,1430.0,1914.0,4334.0,2635.0,4840.0,10463.0,70960.0,5847.0,954.0,5933.0,15808.0,7393.0,70.0,317.0,25.0,68.0,468.0,63.0,45.0,108.0,10503.0,4441.0,165.0,163.0,87.0,1857.0,149.0,19.0,318.0,485.0,82.0,2744.0,3220.0,1283.0,276.0,2502.0,231.0,900.0,2552.0,270.0,586.0,323.0,788.0,9051.0,69247.0,165.0,94.0,52.0,108.0,141.0,903.0,51.0,15.0,2203.0,2428.0,315.0,159.0,54.0,94.0,146.0,637.0,269.0,186.0,237.0,746789.0,139626.0,121244.0
units_total_series,5977.0,52167.0,3901.0,8482.0,17614.0,13508.0,20995.0,51657.0,252977.0,34921.0,3095.0,47005.0,70616.0,28674.0,222.0,1508.0,1148.0,446.0,7934.0,600.0,124.0,420.0,40042.0,14365.0,1135.0,557.0,171.0,5288.0,651.0,207.0,655.0,2218.0,1788.0,17214.0,9644.0,5862.0,2020.0,16496.0,1017.0,6984.0,8749.0,780.0,2028.0,4608.0,1975.0,28952.0,242474.0,859.0,1013.0,1869.0,815.0,1037.0,3511.0,402.0,81.0,9995.0,5778.0,2871.0,583.0,454.0,139.0,598.0,1945.0,893.0,977.0,2594.0,2439443.0,582915.0,506845.0
units_one_detached,4106.0,36963.0,2599.0,5818.0,12542.0,10486.0,15976.0,38505.0,133369.0,27191.0,2079.0,36999.0,48200.0,20735.0,182.0,914.0,1080.0,213.0,7240.0,434.0,84.0,317.0,27055.0,9676.0,922.0,399.0,98.0,3560.0,525.0,187.0,438.0,1641.0,1763.0,10196.0,6352.0,3696.0,1577.0,11594.0,806.0,5253.0,5488.0,564.0,1286.0,3767.0,1450.0,15104.0,125507.0,684.0,955.0,1762.0,752.0,925.0,2592.0,377.0,62.0,6308.0,3868.0,2546.0,491.0,415.0,106.0,487.0,1422.0,672.0,749.0,2218.0,1642085.0,374833.0,320975.0
units_one_attached,50.0,1610.0,11.0,67.

In [320]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']

In [321]:
data = data.transpose().reset_index()

In [322]:
data.head()

,NAME,poverty_total_bysexbyage_series,poverty_belowlevel,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhincome_median,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%ownercost50+_wmortgage,housingcost_total%ownercostwomortgage_series,housingcost_%ownercost30to34.9_womortgage,housingcost_%ownercost35to39.9_womortgage,housingcost_%ownercost40to49.9_womortgage,housingcost_%ownercost50+_womortgage,housingcost_total_rent%hhincome_series,housingcost_%rentercost30to34.9,housingcost_%rentercost35to39.9,housingcost_%rentercost40to49.9,housingcost_%rentercost50+,housingcost_medvalue_ownerocc,housingcost_mediangrossrent_renteroccupied,structures_total_yearbuilt_series,structures_built1999to2000,structures_built1995to1998,structures_built1990to1994,structures_built1980to1989,structures_built1970to1979,structures_built1960to1969,structures_built1950to1959,structures_built1940to1949,structures_built1939orearlier,structures_medianyearbuilt,commute_total_meansoftransportationtowork_series,commute_cartruckvan,commute_cartruckvan_drovealone,commute_cartruckvan_carpooled,commute_publictransportation,commute_publictransportation_bus,commute_publictransportation_subwayorelevatedrail,commute_publictransportation_longdistancetrainorcommuterrail,commute_publictransportation_lightrailstreetcarortrolley,commute_publictransportation_ferryboat,commute_bicycle,commute_walk,commute_taxicab,commute_motorcycle,commute_othermeans,commute_workedfromhome,vehicles_tenurebyvehicles_total_series,vehicles_ownerocc_novehicle,vehicles_ownerocc_1vehicle,vehicles_ownerocc_2vehicles,vehicles_ownerocc_3vehicles,vehicles_ownerocc_4vehicles,vehicles_ownerocc_5+vehicles,vehicles_renterocc_novehicle,vehicles_renterocc_1vehicle,vehicles_renterocc_2vehicles,vehicles_renterocc_3vehicles,vehicles_renterocc_4vehicles,vehicles_renterocc_5+vehicles,foreignborn_total,fb_europe,fb_eur_northern,fb_eur_n_ireland,fb_eur_n_uk,fb_eur_n_other,fb_eur_western,fb_eur_w_austria,fb_eur_w_france,fb_eur_w_germany,fb_eur_w_netherlands,fb_eur_w_other,fb_eur_southern,fb_eur_s_greece,fb_eur_s_italy,fb_eur_s_portugal,fb_eur_s_spain,fb_eur_s_other,fb_eur_eastern,fb_eur_e_belarus,fb_eur_e_bosniaandherzegovina,fb_eur_e_czechoslovakia,fb_eur_e_hungary,fb_eur_e_poland,fb_eur_e_romania,fb_eur_e_russia,fb_eur_e_ukraine,fb_eur_e_other,fb_eur_nec,fb_asia,fb_as_eastern,fb_as_e_china,fb_as_e_chinanohongkongortaiwan,fb_as_e_hongkong,fb_as_e_taiwan,fb_as_e_japan,fb_as_e_korea,fb_as_e_other,fb_as_southcentral,fb_as_sc_afghanistan,fb_as_sc_bangladesh,fb_as_sc_india,fb_as_sc_iran,fb_as_sc_pakistan,fb_as_sc_other,fb_as_southeastern,fb_as_se_cambodia,fb_as_se_indonesia,fb_as_se_laos,fb_as_se_malaysia,fb_as_se_philippines,fb_as_se_thailand,fb_as_se_vietnam,fb_as_se_other,fb_as_western,fb_as_w_armenia,fb_as_w_iraq,fb_as_w_israel,fb_as_w_jordan,fb_as_w_lebanon,fb_as_w_syria,fb_as_w_turkey,fb_as_w_other,fb_as_nec,fb_africa,fb_af_eastern,fb_af_e_ethiopia,fb_af_e_other,fb_af_middle,fb_af_northern,fb_af_n_egypt,fb_af_n_other,fb_af_southern,fb_af_s_southafrica,fb_af_s_other,fb_af_western,fb_af_w_ghana,fb_af_w_nigeria,fb_af_w_sierraleone,fb_af_w_other,fb_af_nec,fb_oceania,fb_oc_australianewzealandsubregion,fb_oc_australianewzealandsubregion_australia,fb_oc_australianewzealandsubregion_other,fb_oc_micronesia,fb_oc_nec,fb_americas,fb_am_latinamerica,fb_am_la_caribbean,fb_am_la_cbb_barbados,fb_am_la_cbb_cuba,fb_am_la_cbb_dominicanrepublic,fb_am_la_cbb_haiti,fb_am_la_cbb_jamaica,fb_am_la_cbb_trinidadandtobago,fb_am_la_cbb_other,fb_am_la_centralamerica,fb_am_la_centr_costarica,fb_am_la_centr_elsalvador,fb_am_la_centr_guatemala,fb_am_la_centr_honduras,fb_am_la_centr_mexico,fb_am_la_centr_nicaragua,fb

In [323]:
data.to_csv('../data/SF3.csv', index = False)

# Come back for hartsville/trousdale.... not there for some reason